# Solar notebook example

This notebook is a simple example of how to use the `solar_maintenance_app` to monitor the energy production of a PV system. 

1. To successfully execute this notebook, you need to have the following required environment variables which can be saved in a `.env` file:
    - `WEATHER_SERVICE_ADDRESS`: The address of the weather service.
    - `REPORTING_SERVICE_ADDRESS`: The address of the reporting service.
    - `REPORTING_API_KEY`: The API key for the reporting service.
2. The list `all_sites` must also be set up as shown in the example.
3. You can adjust the parameter values in the `USER_REQUEST` dictionary as needed. For more details on the parameters refer to `config.py`

In [1]:
import datetime
import os
from dataclasses import asdict, dataclass, field

from dotenv import load_dotenv

from frequenz.lib.notebooks.solar.maintenance.solar_maintenance_app import run_workflow

In [2]:
# Load environment variables from the .env file
load_dotenv()

True

In [3]:
"""Dataclass for the configuration of the client site."""

@dataclass
class ClientSiteInfo:
    """Detailed configuration for specific client site information."""

    latitude: float = field(
        metadata={
            "description": "Geographic latitude of the site.",
            "required": True,
        },
    )

    longitude: float = field(
        metadata={
            "description": "Geographic longitude of the site.",
            "required": True,
        },
    )

    peak_power_watts: float = field(
        metadata={
            "description": "The peak power (solar panels) in Watts.",
            "required": True,
        },
    )

    rated_power_watts: float = field(
        metadata={
            "description": "The rated power (PV inverters) in Watts.",
            "required": True,
        },
    )

    efficiency: float = field(
        default=0.85,
        metadata={
            "description": "An estimate of the overall system efficiency.",
            "required": False,
            "validate": lambda x: 0.0 <= x <= 1.0,
        },
    )

    altitude: float = field(
        default=0.0,
        metadata={
            "description": "Geographic altitude of the site.",
            "required": False,  # only used for the pvlib simulation
        },
    )


@dataclass
class SiteConfig:
    """Main configuration for a client site entry, including component and site details."""

    mid: int = field(
        metadata={
            "description": "The microgrid ID of the client site",
            "required": True,
        },
    )

    component_ids: list[int] = field(
        metadata={
            "description": "List of component IDs associated with the site",
            "required": True,
        },
    )

    client_site_info: ClientSiteInfo = field(
        metadata={
            "description": "Detailed configuration for client site information",
            "required": True,
        },
    )


# define all sites here
all_sites = [
    # example usage:
    # asdict(
    #     SiteConfig(
    #         mid=0,
    #         component_ids=[0, 1],
    #         client_site_info=ClientSiteInfo(
    #             latitude=0.0,
    #             longitude=0.0,
    #             peak_power_watts=0.0,
    #             rated_power_watts=0.0,
    #             efficiency=0.0,
    #         ),
    #     )
    # ),
]

In [4]:
# example user request (1 year of data)
end_time = datetime.datetime.now().astimezone(datetime.timezone.utc)
start_time = end_time - datetime.timedelta(days=364)

USER_REQUEST = {
    "weather_service_address": os.environ.get("WEATHER_SERVICE_ADDRESS"),
    "reporting_service_address": os.environ.get("REPORTING_SERVICE_ADDRESS"),
    "microgrid_ids": [x["mid"] for x in all_sites],
    "component_ids": [x["component_ids"] for x in all_sites],
    "client_site_info": [x["client_site_info"] for x in all_sites],
    "time_zone": "Europe/Berlin",
    "end_timestamp": end_time,
    "start_timestamp": start_time,
    "baseline_models": [
        "7-day MA",
        "7-day sampled MA",
        "weather-based-forecast",
    ],
    "large_resample_period_seconds": 3600,
    "small_resample_period_seconds": 2,
    "language": "en",
}

In [5]:
if len(all_sites) > 0:
    plot_data = await run_workflow(user_config_changes=USER_REQUEST)